# AETHER STT — phase 1 (CTC branch) training notebook

Clones `aether-v3` from GitHub and runs the CTC-only pipeline: frozen Mimi encoder → semantic codes → `AetherSpeech` transformer → CTC head, on LibriSpeech.

The dummy-dataset smoke test (pipeline sanity check on a handful of utterances) already ran and passed locally - no need to repeat it here. This notebook goes straight to the real LibriSpeech run.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_DIR = "aether-v3"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
import importlib.util
import subprocess
import sys


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


# Most GPU notebook images already ship a CUDA-matched torch build — don't
# clobber it. Only install if genuinely missing.
if importlib.util.find_spec("torch") is None:
    pip_install("torch")
if importlib.util.find_spec("torchaudio") is None:
    pip_install("torchaudio")

pip_install(
    "transformers>=5.17",  # <5.17 lacks MimiModel.get_audio_codes_mask - see mimi_wrapper.py
    "datasets>=2.19,<4.0",  # >=4.0 requires torchcodec + system ffmpeg for Audio decoding
    "soundfile",
    "librosa",  # datasets<4.0's Audio decode path needs this alongside soundfile
    "jiwer",
    "pyyaml",
    "numpy",
    "tqdm",
)

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import torch

from aether_v3.config import load_config

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
print("using device:", DEVICE)
assert DEVICE == "cuda", "No GPU visible on this VM - check the Colab session's accelerator."

## Real run: LibriSpeech clean-100 + clean-360

**This downloads the full configured splits, not a sample.** Per `configs/ctc_base.yaml`: `train.100` (~6GB) + `train.360` (~24GB) for training, plus `dev-clean`/`dev-other`/`test-clean`/`test-other` (a few hundred MB each) for eval - roughly **30-35GB total**, cached under `data_cache/ctc_base/` on the VM's local disk. Re-running this cell is safe (already-cached splits are skipped; a config change is detected and raises instead of silently reusing stale data - see `mimi_cache.py`).

If you want to sanity-check on less data/compute before committing to the full 460h, edit `configs/ctc_base.yaml`'s `data.train_splits` down to just `["clean/train.100"]` before running this cell (and delete `data_cache/ctc_base/` if you already ran it with the larger split).

Meant to run unattended on a rented GPU for a long time afterward - check `train.max_steps` / `train.batch_size` in the config for your hardware before launching training below.

In [ ]:
from aether_v3.data.mimi_cache import prepare_cache

real_config = load_config("configs/ctc_base.yaml")
prepare_cache(real_config, device=DEVICE)

### Train

**Single GPU:** run the Python cell below.

**Multiple GPUs on this machine:** don't use the Python cell — use the shell cell instead (`torchrun` spawns its own processes; the training loop auto-detects its environment variables and switches to DDP with no code changes).

In [ ]:
from aether_v3.training.train_ctc import run_training

run_training(real_config)

In [ ]:
# Multi-GPU alternative to the cell above — edit nproc_per_node, then run this
# cell instead of the plain `run_training(real_config)` call.
# NPROC = 4
# !torchrun --nproc_per_node={NPROC} -m aether_v3.training.train_ctc --config configs/ctc_base.yaml

## Monitor training

Re-run this cell any time (even from a second notebook while the cell above is still training) to see the latest loss/WER/CER curves from `log.jsonl`.

Watch WER/CER, not eval loss - CTC-infeasible examples (target longer than the Mimi frame count) get `zero_infinity`-clamped to ~0 loss regardless of how well the model is actually doing, so eval loss alone is misleading.

In [ ]:
import json

import matplotlib.pyplot as plt

with open(f"{real_config.train.output_dir}/log.jsonl") as f:
    rows = [json.loads(line) for line in f]

train_rows = [r for r in rows if "loss" in r and "eval_loss" not in r]
eval_rows = [r for r in rows if "eval_cer" in r]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([r["step"] for r in train_rows], [r["loss"] for r in train_rows])
axes[0].set_title("train loss")
axes[0].set_xlabel("step")

axes[1].plot([r["step"] for r in eval_rows], [r["eval_cer"] for r in eval_rows], label="CER")
axes[1].plot([r["step"] for r in eval_rows], [r["eval_wer"] for r in eval_rows], label="WER")
axes[1].set_title("dev CER / WER")
axes[1].set_xlabel("step")
axes[1].legend()
plt.show()

if eval_rows:
    best = min(eval_rows, key=lambda r: r["eval_cer"])
    print("best eval so far:", best)